# CDD-11-30: A1/A2 CoT-NAFNet calibration

This notebook compares the bottleneck adapter (A1) with bottleneck + skip gates (A2). Both runs use the selected SIDD32 checkpoint, the same five-epoch controls as A0, two T4 GPUs through DDP, and full-frame validation/evaluation. Expected Kaggle runtime is approximately 15–25 minutes for both runs.

Only the existing `cdd-11-30` and `nafnetmodel` Kaggle inputs are required. GoPro and SIDD image datasets are not required.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/HoangKhanhTung0111/CoT-restoration.git"
REPO_DIR = Path("/kaggle/working/CoT-restoration")
if REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Commit:", COMMIT)
print("CWD:", Path.cwd())

In [ ]:
CDD11_ROOT = Path("/kaggle/input/datasets/mintesnotfikir/cdd-11-30")
PRETRAINED_ROOT = Path("/kaggle/input/datasets/hoangkhanhtung/nafnetmodel")
EXPERIMENTS_ROOT = Path("/kaggle/working/experiments_a1_a2")
CONFIG = Path("configs/calibration_a1_a2.json")
assert CDD11_ROOT.is_dir(), f"Missing CDD-11 input: {CDD11_ROOT}"
assert PRETRAINED_ROOT.is_dir(), f"Missing pretrained input: {PRETRAINED_ROOT}"
assert CONFIG.is_file(), f"Missing config: {CONFIG}"
gpu_names = subprocess.check_output([
    "nvidia-smi", "--query-gpu=name", "--format=csv,noheader"
], text=True).strip().splitlines()
assert len(gpu_names) == 2, f"Select the Kaggle 2xT4 accelerator; found: {gpu_names}"
print("GPUs:", gpu_names)
print("CDD-11:", CDD11_ROOT)
print("Pretrained files:", sorted(path.name for path in PRETRAINED_ROOT.glob("*.pth")))

In [ ]:
# Recheck data pairs, leakage, and exact SIDD32 checkpoint compatibility.
subprocess.run([
    "python", "-m", "hybrid_cot_nafnet.audit_kaggle",
    "--data-root", str(CDD11_ROOT),
    "--pretrained-root", str(PRETRAINED_ROOT),
    "--output", "/kaggle/working/cot_nafnet_audit/audit_a1_a2.json",
], check=True)

In [ ]:
# Safety switch: first Run All prints and validates the exact commands only.
RUN_ABLATIONS = False
RUN_NAMES = [
    "a1_bottleneck_adapter_sidd32_seed42_5ep",
    "a2_bottleneck_skip_sidd32_seed42_5ep",
]

In [ ]:
runner_command = [
    "python", "-m", "hybrid_cot_nafnet.run_ablation",
    "--config", str(CONFIG),
    "--data-root", str(CDD11_ROOT),
    "--experiments-root", str(EXPERIMENTS_ROOT),
    "--nproc-per-node", "2",
    "--runs", *RUN_NAMES,
]
if RUN_ABLATIONS:
    subprocess.run(runner_command, check=True)
else:
    subprocess.run([*runner_command, "--dry-run"], check=True)
    print("Dry run complete. Set RUN_ABLATIONS = True and rerun from the safety-switch cell.")

In [ ]:
# Display the comparison and make one small download containing logs/metrics only.
import zipfile
from IPython.display import FileLink, FileLinks, display

summary_csv = EXPERIMENTS_ROOT / "ablation_summary.csv"
if summary_csv.is_file():
    import pandas as pd
    display(pd.read_csv(summary_csv))
    archive = Path("/kaggle/working/a1_a2_lightweight_results.zip")
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as output_zip:
        for path in sorted(EXPERIMENTS_ROOT.rglob("*")):
            if path.is_file() and path.suffix.lower() not in {".pt", ".png", ".jpg", ".jpeg"}:
                output_zip.write(path, path.relative_to(EXPERIMENTS_ROOT))
    print("Download this archive and send it back for analysis:")
    display(FileLink(str(archive)))
else:
    print("No completed ablation summary yet.")

if EXPERIMENTS_ROOT.is_dir():
    display(FileLinks(str(EXPERIMENTS_ROOT)))